# 09 - Trio-Centric Clustering and Enrichment

Unsupervised exploration of cohort structure using a **trio/proband-centric** approach.

Why this design:
- HPO terms are usually recorded for probands, not healthy parents.
- Clustering at per-sample level can bias results due to structural parent HPO missingness.
- We therefore build **one row per trio** with genotype/inheritance summaries plus proband HPO features.

Outputs:
- Cluster assignments per trio
- PCA embedding
- Cluster-level gene enrichment
- Cluster-level proband HPO enrichment

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.trio_loader import TrioLoader

DATA_DIR = ROOT / 'data'
REPORTS_DIR = ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

IN_PARQUET = DATA_DIR / 'trio_variants.parquet'
TRIOS = DATA_DIR / 'trios.tsv'
PHENO = DATA_DIR / 'phenotypes.tsv'

OUT_CLUSTER = REPORTS_DIR / 'trio_cluster_assignments.csv'
OUT_EMBED = REPORTS_DIR / 'trio_embedding_pca.csv'
OUT_GENE_ENRICH = REPORTS_DIR / 'cluster_gene_enrichment.csv'
OUT_HPO_ENRICH = REPORTS_DIR / 'cluster_hpo_enrichment.csv'

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

print('ROOT:', ROOT)

In [ ]:
if IN_PARQUET.exists():
    df = pd.read_parquet(IN_PARQUET)
    print(f'Loaded {IN_PARQUET.name}: {df.shape}')
else:
    loader = TrioLoader(trios_path=TRIOS, phenotypes_path=PHENO, data_dir=DATA_DIR)
    df = loader.load_all()
    df.to_parquet(IN_PARQUET, index=False)
    print(f'Generated {IN_PARQUET.name}: {df.shape}')

if 'trio_id' not in df.columns:
    raise ValueError('Expected trio_id in assembled dataframe.')

print('Trio count:', df['trio_id'].nunique())
display(df.head(3))

## 1. Build Trio-Level Feature Table

Features include:
- Variant burden (total variants, VUS ratio, pathogenic ratio)
- Inheritance pattern composition
- Gene burden profile (top recurrent genes in cohort)
- Proband HPO profile (counts + top HPO indicators)

In [ ]:
work = df.copy()

cls = work.get('Classification', pd.Series('', index=work.index)).fillna('').astype(str).str.lower().str.strip()
work['_is_vus'] = cls.eq('uncertain significance').astype(int)
work['_is_path_like'] = cls.isin(['pathogenic', 'likely pathogenic']).astype(int)
work['_is_benign_like'] = cls.isin(['benign', 'likely benign']).astype(int)

grp = work.groupby('trio_id', dropna=False)
trio_base = pd.DataFrame({
    'variant_count': grp.size(),
    'vus_count': grp['_is_vus'].sum(),
    'path_like_count': grp['_is_path_like'].sum(),
    'benign_like_count': grp['_is_benign_like'].sum(),
}).reset_index()

trio_base['vus_ratio'] = trio_base['vus_count'] / trio_base['variant_count'].clip(lower=1)
trio_base['path_like_ratio'] = trio_base['path_like_count'] / trio_base['variant_count'].clip(lower=1)

if 'inheritance_mode' in work.columns:
    inh = pd.crosstab(work['trio_id'], work['inheritance_mode'].fillna('unknown'))
    inh = inh.div(inh.sum(axis=1), axis=0).fillna(0.0)
    inh.columns = [f'inh_frac_{c}' for c in inh.columns]
    inh = inh.reset_index()
    trio_base = trio_base.merge(inh, on='trio_id', how='left')

if 'Gene' in work.columns:
    top_genes = work['Gene'].fillna('UNKNOWN').astype(str).value_counts().head(30).index.tolist()
    gene_sub = work[work['Gene'].fillna('UNKNOWN').astype(str).isin(top_genes)].copy()
    gene_sub['_gene'] = gene_sub['Gene'].fillna('UNKNOWN').astype(str)
    gmat = pd.crosstab(gene_sub['trio_id'], gene_sub['_gene'])
    gmat = gmat.div(gmat.sum(axis=1), axis=0).fillna(0.0)
    gmat.columns = [f'gene_frac_{c}' for c in gmat.columns]
    gmat = gmat.reset_index()
    trio_base = trio_base.merge(gmat, on='trio_id', how='left')

hpo_series = grp['proband_phenotype_hpo'].first() if 'proband_phenotype_hpo' in work.columns else pd.Series('', index=grp.size().index)
hpo_series = hpo_series.fillna('').astype(str)

trio_base['proband_hpo_count'] = hpo_series.apply(lambda s: 0 if not s.strip() else len([x for x in s.split(';') if x.strip()])).values

all_hpo_terms = hpo_series.apply(lambda s: [t.strip() for t in s.split(';') if t.strip()])
hpo_counts = pd.Series([t for row in all_hpo_terms for t in row]).value_counts()
top_hpo = hpo_counts.head(12).index.tolist()

for term in top_hpo:
    trio_base[f'hpo_has_{term}'] = all_hpo_terms.apply(lambda lst: int(term in lst)).values

trio_features = trio_base.fillna(0).copy()

print('Trio feature table shape:', trio_features.shape)
display(trio_features.head(5))

## 2. PCA Embedding and KMeans Clustering

We select K using silhouette score over K=2..6 (subject to cohort size).

In [ ]:
if trio_features['trio_id'].nunique() < 3:
    raise ValueError('Need at least 3 trios for meaningful clustering.')

X = trio_features.drop(columns=['trio_id']).copy()
X = X.apply(pd.to_numeric, errors='coerce').fillna(0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_max = min(6, len(trio_features) - 1)
candidate_k = [k for k in range(2, k_max + 1)]
if not candidate_k:
    candidate_k = [2]

sil_scores = []
for k in candidate_k:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_tmp = km_tmp.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels_tmp) if len(set(labels_tmp)) > 1 else -1
    sil_scores.append((k, score))

best_k, best_sil = max(sil_scores, key=lambda x: x[1])
print('Silhouette by K:', sil_scores)
print('Chosen K:', best_k, '| silhouette:', round(float(best_sil), 4))

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=50)
cluster = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
emb = pca.fit_transform(X_scaled)

cluster_df = trio_features[['trio_id']].copy()
cluster_df['cluster'] = cluster
cluster_df['pca1'] = emb[:, 0]
cluster_df['pca2'] = emb[:, 1]

print('Explained variance ratio:', pca.explained_variance_ratio_.round(4).tolist())
display(cluster_df.sort_values(['cluster', 'trio_id']).head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for c in sorted(cluster_df['cluster'].unique()):
    sub = cluster_df[cluster_df['cluster'] == c]
    ax.scatter(sub['pca1'], sub['pca2'], s=70, alpha=0.85, label=f'cluster {c}')
for _, row in cluster_df.iterrows():
    ax.text(row['pca1'], row['pca2'], str(row['trio_id']), fontsize=8, alpha=0.75)
ax.set_title('PCA embedding of trio-level features')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()

ks = [k for k, _ in sil_scores]
ss = [s for _, s in sil_scores]
axes[1].plot(ks, ss, marker='o')
axes[1].set_title('Silhouette score by K')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')

plt.tight_layout()
plt.show()

## 3. Cluster Enrichment

We test which genes and proband HPO terms are overrepresented in each cluster
relative to the rest of the cohort (simple log2 enrichment ratio).

In [ ]:
work2 = work.merge(cluster_df[['trio_id', 'cluster']], on='trio_id', how='left')

gene_enrich_rows = []
if 'Gene' in work2.columns:
    gene_s = work2['Gene'].fillna('UNKNOWN').astype(str)
    frequent_genes = gene_s.value_counts().head(80).index.tolist()
    work_gene = work2[gene_s.isin(frequent_genes)].copy()
    work_gene['_Gene'] = work_gene['Gene'].fillna('UNKNOWN').astype(str)

    for c in sorted(work_gene['cluster'].dropna().unique()):
        in_c = work_gene[work_gene['cluster'] == c]
        out_c = work_gene[work_gene['cluster'] != c]
        in_total = max(1, len(in_c))
        out_total = max(1, len(out_c))

        in_cnt = in_c['_Gene'].value_counts()
        out_cnt = out_c['_Gene'].value_counts()

        all_genes = sorted(set(in_cnt.index) | set(out_cnt.index))
        for g in all_genes:
            a = int(in_cnt.get(g, 0))
            b = int(out_cnt.get(g, 0))
            in_rate = (a + 0.5) / (in_total + 1.0)
            out_rate = (b + 0.5) / (out_total + 1.0)
            gene_enrich_rows.append({
                'cluster': int(c),
                'gene': g,
                'in_cluster_count': a,
                'out_cluster_count': b,
                'log2_enrichment': float(np.log2(in_rate / out_rate))
            })

gene_enrich = pd.DataFrame(gene_enrich_rows)
if not gene_enrich.empty:
    gene_enrich = gene_enrich.sort_values(['cluster', 'log2_enrichment'], ascending=[True, False])

hpo_enrich_rows = []
trio_hpo = pd.DataFrame({
    'trio_id': hpo_series.index.astype(str),
    'hpo_str': hpo_series.values
}).merge(cluster_df[['trio_id', 'cluster']], on='trio_id', how='left')

trio_hpo['hpo_terms'] = trio_hpo['hpo_str'].fillna('').astype(str).apply(lambda s: [t.strip() for t in s.split(';') if t.strip()])

all_terms = pd.Series([t for row in trio_hpo['hpo_terms'] for t in row]).value_counts()
freq_terms = all_terms.head(40).index.tolist()

for c in sorted(trio_hpo['cluster'].dropna().unique()):
    in_c = trio_hpo[trio_hpo['cluster'] == c]
    out_c = trio_hpo[trio_hpo['cluster'] != c]
    in_total = max(1, len(in_c))
    out_total = max(1, len(out_c))

    for term in freq_terms:
        a = int(in_c['hpo_terms'].apply(lambda lst: term in lst).sum())
        b = int(out_c['hpo_terms'].apply(lambda lst: term in lst).sum())
        in_rate = (a + 0.5) / (in_total + 1.0)
        out_rate = (b + 0.5) / (out_total + 1.0)
        hpo_enrich_rows.append({
            'cluster': int(c),
            'hpo_term': term,
            'in_cluster_count': a,
            'out_cluster_count': b,
            'log2_enrichment': float(np.log2(in_rate / out_rate))
        })

hpo_enrich = pd.DataFrame(hpo_enrich_rows)
if not hpo_enrich.empty:
    hpo_enrich = hpo_enrich.sort_values(['cluster', 'log2_enrichment'], ascending=[True, False])

display(gene_enrich.groupby('cluster').head(10) if not gene_enrich.empty else pd.DataFrame())
display(hpo_enrich.groupby('cluster').head(10) if not hpo_enrich.empty else pd.DataFrame())

In [ ]:
cluster_df.sort_values(['cluster', 'trio_id']).to_csv(OUT_CLUSTER, index=False)
cluster_df.sort_values(['cluster', 'trio_id']).to_csv(OUT_EMBED, index=False)

if 'gene_enrich' in locals() and not gene_enrich.empty:
    gene_enrich.to_csv(OUT_GENE_ENRICH, index=False)

if 'hpo_enrich' in locals() and not hpo_enrich.empty:
    hpo_enrich.to_csv(OUT_HPO_ENRICH, index=False)

print('Saved cluster assignments:', OUT_CLUSTER)
print('Saved PCA embedding      :', OUT_EMBED)
print('Saved gene enrichment   :', OUT_GENE_ENRICH if ('gene_enrich' in locals() and not gene_enrich.empty) else 'no gene enrichment rows')
print('Saved HPO enrichment    :', OUT_HPO_ENRICH if ('hpo_enrich' in locals() and not hpo_enrich.empty) else 'no HPO enrichment rows')